In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BorsaBot — 05 | CPCV Backtest & Model Değerlendirmesi
# ═══════════════════════════════════════════════════════════════════
# Önceki notebookların çalışmış olması gerekir.

In [ ]:
# ─────────────────────────────────────────────────────────────────
import itertools, math
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt

def cpcv_splits(n: int, n_groups: int = 6, n_test: int = 2,
                embargo_pct: float = 0.01):
    """
    C(n_groups, n_test) adet (train_idx, test_idx) üretir.
    Embargo: her test bloğundan önce embargo_n sample temizlenir.
    """
    group_size = n // n_groups
    borders    = [i * group_size for i in range(n_groups)] + [n]
    groups     = [list(range(borders[i], borders[i+1]))
                  for i in range(n_groups)]
    embargo_n  = max(1, int(n * embargo_pct))

    for test_ids in itertools.combinations(range(n_groups), n_test):
        test_set  = set(j for g in test_ids for j in groups[g])
        train_set = set(j for g in range(n_groups)
                        if g not in test_ids for j in groups[g])

        # Embargo: test başlangıcından önceki embargo_n sample'ı train'den çıkar
        embargo_zone = set()
        for pos in test_set:
            for off in range(1, embargo_n + 1):
                if (pos - off) >= 0 and (pos - off) not in test_set:
                    embargo_zone.add(pos - off)
        train_set -= embargo_zone

        if train_set and test_set:
            yield (np.array(sorted(train_set)),
                   np.array(sorted(test_set)))

In [ ]:
# ─────────────────────────────────────────────────────────────────
FEE_BPS      = 7.0   # toplam komisyon + slippage (bps)
INITIAL_CAP  = 100_000.0

def simulate_path(signals: np.ndarray, prices: np.ndarray) -> dict:
    """
    Basit bar-by-bar simülasyon.
    signals: {-1, 0, +1}  prices: kapanış fiyatları
    """
    cap   = INITIAL_CAP
    pos   = 0
    equit = [cap]
    rets  = []
    cost  = FEE_BPS / 10_000

    for i in range(1, len(signals)):
        p_prev, p_now = prices[i-1], prices[i]
        sig = signals[i]

        # M2M PnL
        if pos != 0 and p_prev > 0:
            bar_ret = (p_now - p_prev) / p_prev * pos
            cap    *= (1 + bar_ret)

        # Pozisyon değişimi
        if sig != pos:
            cap  *= (1 - abs(sig - pos) * cost)   # işlem maliyeti
            pos   = int(sig)

        equit.append(cap)
        rets.append(cap / equit[-2] - 1 if equit[-2] > 0 else 0.0)

    equity  = np.array(equit)
    returns = np.array(rets)
    std     = returns.std() + 1e-9
    sharpe  = float(returns.mean() / std * np.sqrt(24 * 365))  # saatlik → yıllık
    roll_max = np.maximum.accumulate(equity)
    mdd     = float(((equity - roll_max) / (roll_max + 1e-9)).min())
    return {"sharpe": sharpe, "mdd": mdd, "equity": equity, "returns": returns}

In [ ]:
# ─────────────────────────────────────────────────────────────────
N_GROUPS = 6
N_TEST   = 2

def run_cpcv(sym: str):
    df      = labeled[sym].copy()
    feat    = df[FEAT_COLS].values
    y_true  = df["label"].map(LABEL_MAP).values
    close   = ohlcv[sym]["close"].reindex(df.index).ffill().values
    prim    = primary_models[sym]
    meta    = meta_models[sym]
    mfeat   = [c for c in FEAT_COLS + ["primary_pred","primary_conf"]
               if c in FEAT_COLS + ["primary_pred","primary_conf"]]

    sharpes, mdds, equity_paths = [], [], []

    for fold, (tr_idx, te_idx) in enumerate(
            cpcv_splits(len(df), N_GROUPS, N_TEST)):

        X_tr = feat[tr_idx]
        y_tr = y_true[tr_idx]
        X_te = feat[te_idx]

        # Primary model (fold bazında tekrar fit — gerçek CPCV gereği)
        prim_fold = type(prim)(**prim.get_params())
        prim_fold.set_params(verbosity=0)
        try:
            prim_fold.fit(X_tr, y_tr)
        except Exception:
            continue

        # Sinyaller
        pred  = prim_fold.predict(X_te)
        proba = prim_fold.predict_proba(X_te)
        conf  = proba.max(axis=1)

        # Meta filtresi
        X_meta = np.column_stack([
            X_te,
            np.vectorize(LABEL_RMAP.get)(pred),
            conf,
        ])
        meta_pred = meta.predict_proba(X_meta)[:, 1]  # P(correct)
        THRESHOLD = 0.55
        sig_raw   = np.vectorize(LABEL_RMAP.get)(pred).astype(float)
        signals   = np.where(meta_pred >= THRESHOLD, sig_raw, 0.0)

        # Simülasyon
        prices = close[te_idx]
        result = simulate_path(signals, prices)
        sharpes.append(result["sharpe"])
        mdds.append(result["mdd"])
        equity_paths.append(result["equity"])

    if not sharpes:
        print(f"[{sym}] ⚠ Hiç başarılı fold yok!")
        return

    sarr   = np.array(sharpes)
    psr    = float(norm.cdf(sarr.mean() / (sarr.std() + 1e-9)))

    print(f"\n{'='*55}")
    print(f"  {sym} — CPCV Sonuçları  ({len(sharpes)} path)")
    print(f"{'='*55}")
    print(f"  Ortalama Sharpe : {sarr.mean():.3f}  ± {sarr.std():.3f}")
    print(f"  Min / Max Sharpe: {sarr.min():.3f}  / {sarr.max():.3f}")
    print(f"  Ortalama MDD    : {np.mean(mdds)*100:.1f}%")
    print(f"  PSR (>0 prob)   : {psr:.3f}")

    # Sharpe dağılımı
    plt.figure(figsize=(8, 3))
    plt.hist(sarr, bins=15, color="#4C72B0", edgecolor="white")
    plt.axvline(0, color="red", linestyle="--", label="SR=0")
    plt.axvline(sarr.mean(), color="green", linestyle="-", label=f"Mean={sarr.mean():.2f}")
    plt.title(f"{sym} — CPCV Sharpe Dağılımı  PSR={psr:.2f}")
    plt.xlabel("Sharpe Ratio (yıllık)")
    plt.legend(); plt.tight_layout()
    plt.savefig(f"{CFG['out_dir']}/{sym}_sharpe_dist.png", dpi=80)
    plt.close()

    # Equity path görselleştirmesi
    plt.figure(figsize=(10, 4))
    for eq in equity_paths:
        norm_eq = eq / eq[0]
        plt.plot(norm_eq, alpha=0.4, linewidth=0.8)
    plt.title(f"{sym} — CPCV Equity Paths (normalize edilmiş)")
    plt.xlabel("Bar"); plt.ylabel("Büyüme")
    plt.tight_layout()
    plt.savefig(f"{CFG['out_dir']}/{sym}_equity_paths.png", dpi=80)
    plt.close()
    print(f"  Grafikler: {CFG['out_dir']}/")

for sym in CFG["symbols"]:
    run_cpcv(sym)

print("\n✓ CPCV Backtest tamamlandı!")

In [ ]:
# ─────────────────────────────────────────────────────────────────
import shutil

# Google Drive mount (opsiyonel — çalıştırmak için yorum satırını kaldır)
# from google.colab import drive
# drive.mount('/content/drive')
# drive_path = "/content/drive/MyDrive/borsabot_models"
# shutil.copytree(CFG["model_dir"], drive_path, dirs_exist_ok=True)
# print(f"✓ Modeller Drive'a kopyalandı: {drive_path}")

# Yerel indirme için ZIP oluştur
shutil.make_archive("/content/borsabot_models", "zip", CFG["model_dir"])
print("✓ /content/borsabot_models.zip oluşturuldu — sağ menüden indir!")